# Results — all four conditions

Pulls together the standardized `*_predictions.csv` files from all four
conditions (ModernBERT nominal, T5Gemma2 nominal LoRA, Gemma4 12B LoRA,
Gemma4 12B prompted 24-shot) and produces:

- a metrics table (QWK, MAE, exact/adjacent accuracy, macro-F1,
  ECE, mean confidence), for both val and test
- a combined calibration table across all four conditions, in 10% confidence
  bands
- a 4-panel confusion matrix figure (test set), zero cells left blank

No training or inference happens here — this notebook only reads the CSVs
each condition's VAL/TEST notebook already wrote.

In [1]:
# %%
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix, roc_auc_score

from metrics import evaluate_predictions

LEVELS = ["A2", "A2+", "B1", "B1+", "B2", "B2+", "C1", "C1+"]
NUM_CLASSES = len(LEVELS)

OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)


CONDITIONS = [
    dict(name="ModernBERT nominal",
         run_dir="C:/Users/coope/Documents/Learner_Writing_Classification/_Gemma_2026/T5Gemma/runs/modernbert_nominal",
         dev_file="dev_predictions.csv", test_file="test_predictions.csv"),
    dict(name="T5Gemma2 nominal (LoRA)",
         run_dir="C:/Users/coope/Documents/Learner_Writing_Classification/_Gemma_2026/T5Gemma/runs/t5gemma2_nominal",
         dev_file="dev_predictions.csv", test_file="test_predictions.csv"),
    dict(name="Gemma 4 12B LoRA",
         run_dir="C:/Users/coope/Documents/Learner_Writing_Classification/_Gemma_2026/runs/gemma4_lora",
         dev_file="val_predictions.csv", test_file="test_predictions.csv"),
    dict(name="Gemma 4 12B prompted (24-shot)",
         run_dir="C:/Users/coope/Documents/Learner_Writing_Classification/_Gemma_2026/runs/gemma4_12b_prompting",
         dev_file="val_few_shot_24_predictions.csv",
         test_file="test_few_shot_24_predictions.csv"),
]

## Loading

Every condition's CSV already has the standardized columns (`gold`, `pred`,
`correct`, `adjacent`, `p_pred`, per-class `p_<level>`, etc.) written by its
own VAL/TEST notebook, so no recomputation is needed here — just read and
aggregate.

In [2]:
def load_preds(run_dir, filename):
    path = os.path.join(run_dir, filename)
    d = pd.read_csv(path)
    return d


def ece(d, n_bins=10):
    b = pd.cut(d["p_pred"], np.linspace(0, 1, n_bins + 1))
    g = d.groupby(b, observed=True)
    gap = (g["correct"].mean() - g["p_pred"].mean()).abs()
    return float((gap * g.size()).sum() / len(d))


RENAME = {"accuracy": "exact_acc", "adjacent_accuracy": "adjacent_acc"}
PREFERRED = ["qwk", "mae", "exact_acc", "adjacent_acc", "f1_macro", "ece", "mean_conf", "n"]


def metrics_row(d):
    m = evaluate_predictions(d["gold"].to_numpy(), d["pred"].to_numpy())
    m = {RENAME.get(k, k): v for k, v in m.items()}
    m["ece"] = ece(d)
    m["mean_conf"] = d["p_pred"].mean()
    m["n"] = len(d)
    return m

## Metrics tables (poster-style)

Same metric set and column order as the poster table, plus ECE and mean
confidence.

In [3]:
def build_summary(split_key):
    cols = {}
    for c in CONDITIONS:
        d = load_preds(c["run_dir"], c[split_key])
        cols[c["name"]] = metrics_row(d)
    table = pd.DataFrame(cols)
    order = [m for m in PREFERRED if m in table.index] + \
            [m for m in table.index if m not in PREFERRED]
    return table.loc[order].round(3)


dev_summary = build_summary("dev_file")
test_summary = build_summary("test_file")

dev_summary.to_csv(os.path.join(OUT_DIR, "dev_metrics_summary.csv"))
test_summary.to_csv(os.path.join(OUT_DIR, "test_metrics_summary.csv"))

print("=== Val ===")
print(dev_summary.to_string())
print()
print("=== Test ===")
print(test_summary.to_string())

=== Val ===
                    ModernBERT nominal  T5Gemma2 nominal (LoRA)  Gemma 4 12B LoRA  Gemma 4 12B prompted (24-shot)
qwk                              0.853                    0.905             0.855                           0.842
mae                              0.579                    0.456             0.576                           0.676
exact_acc                        0.501                    0.574             0.514                           0.431
adjacent_acc                     0.935                    0.973             0.923                           0.905
f1_macro                         0.344                    0.454             0.368                           0.406
ece                              0.043                    0.059             0.065                           0.401
mean_conf                        0.479                    0.632             0.461                           0.832
n                              599.000                  599.000           59

In [4]:
test_summary

,ModernBERT nominal,T5Gemma2 nominal (LoRA),Gemma 4 12B LoRA,Gemma 4 12B prompted (24-shot)
qwk,0.832,0.898,0.838,0.832
mae,0.611,0.462,0.626,0.700
exact_acc,0.485,0.576,0.474,0.401
adjacent_acc,0.922,0.965,0.912,0.904
f1_macro,0.328,0.442,0.323,0.365
ece,0.037,0.042,0.039,0.428
mean_conf,0.473,0.616,0.458,0.829
n,604.000,604.000,604.000,604.000
precision_macro,0.378,0.458,0.327,0.409
recall_macro,0.347,0.444,0.339,0.367


## Bootstrap confidence intervals (test)

Resamples each condition's rows with replacement and recomputes the whole
metrics row per resample, so QWK, MAE, exact/adjacent accuracy, macro-F1, and
ECE all get a CI from the same pass rather than five separate loops. Not
every metric needs to end up in the paper — this just makes the CIs
available to choose from once you decide which comparisons to report.

`mean_conf` and `n` are left out: `n` is fixed, not stochastic, and
`mean_conf` is a descriptive diagnostic rather than a claim being compared
across conditions.

In [5]:
CI_KEYS = ["qwk", "mae", "exact_acc", "adjacent_acc", "f1_macro", "ece"]
N_BOOT = 2000
SEED = 42


def bootstrap_ci(d, n_boot=N_BOOT, seed=SEED, keys=CI_KEYS):
    rng = np.random.default_rng(seed)
    n = len(d)
    boot = {k: np.empty(n_boot) for k in keys}
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        m = metrics_row(d.iloc[idx])
        for k in keys:
            boot[k][b] = m[k]
    return {k: tuple(np.percentile(boot[k], [2.5, 97.5])) for k in keys}


ci_rows = []
for c in CONDITIONS:
    d = load_preds(c["run_dir"], c["test_file"])
    point = metrics_row(d)
    ci = bootstrap_ci(d)
    row = {"condition": c["name"]}
    for k in CI_KEYS:
        lo, hi = ci[k]
        row[k] = round(point[k], 3)
        row[f"{k}_lo"] = round(lo, 3)
        row[f"{k}_hi"] = round(hi, 3)
    ci_rows.append(row)

ci_df = pd.DataFrame(ci_rows).set_index("condition")
ci_df.to_csv(os.path.join(OUT_DIR, "test_metrics_with_ci.csv"))
ci_df

,qwk,qwk_lo,qwk_hi,mae,mae_lo,mae_hi,exact_acc,exact_acc_lo,exact_acc_hi,adjacent_acc,adjacent_acc_lo,adjacent_acc_hi,f1_macro,f1_macro_lo,f1_macro_hi,ece,ece_lo,ece_hi
condition,,,,,,,,,,,,,,,,,,
ModernBERT nominal,0.832,0.810,0.854,0.611,0.556,0.666,0.485,0.444,0.527,0.922,0.901,0.942,0.328,0.292,0.362,0.037,0.021,0.083
T5Gemma2 nominal (LoRA),0.898,0.883,0.912,0.462,0.417,0.512,0.576,0.535,0.618,0.965,0.950,0.978,0.442,0.399,0.481,0.042,0.031,0.088
Gemma 4 12B LoRA,0.838,0.815,0.857,0.626,0.573,0.682,0.474,0.434,0.513,0.912,0.889,0.934,0.323,0.290,0.355,0.039,0.023,0.083
Gemma 4 12B prompted (24-shot),0.832,0.809,0.853,0.700,0.649,0.752,0.401,0.361,0.442,0.904,0.879,0.927,0.365,0.311,0.412,0.428,0.387,0.468


### Formatted for reading / pasting into the paper

`estimate [95% CI lower, upper]`, one row per condition.

In [6]:
def fmt_ci(point, lo, hi):
    return f"{point:.3f} [{lo:.3f}, {hi:.3f}]"


display_rows = {}
for row in ci_rows:
    display_rows[row["condition"]] = {
        k: fmt_ci(row[k], row[f"{k}_lo"], row[f"{k}_hi"]) for k in CI_KEYS
    }

display_df = pd.DataFrame(display_rows)
print(display_df.to_string())

                ModernBERT nominal T5Gemma2 nominal (LoRA)      Gemma 4 12B LoRA Gemma 4 12B prompted (24-shot)
qwk           0.832 [0.810, 0.854]    0.898 [0.883, 0.912]  0.838 [0.815, 0.857]           0.832 [0.809, 0.853]
mae           0.611 [0.556, 0.666]    0.462 [0.417, 0.512]  0.626 [0.573, 0.682]           0.700 [0.649, 0.752]
exact_acc     0.485 [0.444, 0.527]    0.576 [0.535, 0.618]  0.474 [0.434, 0.513]           0.401 [0.361, 0.442]
adjacent_acc  0.922 [0.901, 0.942]    0.965 [0.950, 0.978]  0.912 [0.889, 0.934]           0.904 [0.879, 0.927]
f1_macro      0.328 [0.292, 0.362]    0.442 [0.399, 0.481]  0.323 [0.290, 0.355]           0.365 [0.311, 0.412]
ece           0.037 [0.021, 0.083]    0.042 [0.031, 0.088]  0.039 [0.023, 0.083]           0.428 [0.387, 0.468]


## Calibration table — 10% confidence bands, all four conditions

One long-format table (test set), so it can be filtered/pivoted per
condition or plotted directly. Highest confidence first, matching the
risk-coverage convention used elsewhere.

In [7]:
def calibration_bands(d, name):
    edges = np.arange(0.0, 1.01, 0.1)
    bands = list(zip(edges[:-1], edges[1:],
                     [f"{lo:.1f}–{hi:.1f}" for lo, hi in zip(edges[:-1], edges[1:])]))
    n = len(d)
    rows = []
    for lo, hi, label in reversed(bands):
        s = d[(d["p_pred"] >= lo) & (d["p_pred"] < hi)]
        rows.append(dict(
            condition=name, conf=label, preds=len(s),
            share=len(s) / n if n else 0.0,
            acc=round(s["correct"].mean(), 3) if len(s) else None,
            adj=round(s["adjacent"].mean(), 3) if len(s) else None,
        ))
    return rows


band_rows = []
for c in CONDITIONS:
    d = load_preds(c["run_dir"], c["test_file"])
    band_rows += calibration_bands(d, c["name"])

calibration_df = pd.DataFrame(band_rows)
calibration_df.to_csv(os.path.join(OUT_DIR, "test_calibration_bands.csv"), index=False)

for c in CONDITIONS:
    sub = calibration_df[calibration_df["condition"] == c["name"]]
    print(f"\n{c['name']}")
    print(sub[["conf", "preds", "share", "acc", "adj"]].to_string(index=False))


ModernBERT nominal
   conf  preds    share   acc   adj
0.9–1.0      0 0.000000   NaN   NaN
0.8–0.9      0 0.000000   NaN   NaN
0.7–0.8      3 0.004967 0.667 1.000
0.6–0.7     57 0.094371 0.649 0.982
0.5–0.6    172 0.284768 0.529 0.959
0.4–0.5    233 0.385762 0.511 0.940
0.3–0.4    124 0.205298 0.339 0.839
0.2–0.3     15 0.024834 0.133 0.667
0.1–0.2      0 0.000000   NaN   NaN
0.0–0.1      0 0.000000   NaN   NaN

T5Gemma2 nominal (LoRA)
   conf  preds    share   acc   adj
0.9–1.0      2 0.003311 0.500 1.000
0.8–0.9     53 0.087748 0.642 1.000
0.7–0.8    103 0.170530 0.689 1.000
0.6–0.7    169 0.279801 0.627 0.964
0.5–0.6    155 0.256623 0.542 0.968
0.4–0.5    105 0.173841 0.429 0.924
0.3–0.4     17 0.028146 0.412 0.882
0.2–0.3      0 0.000000   NaN   NaN
0.1–0.2      0 0.000000   NaN   NaN
0.0–0.1      0 0.000000   NaN   NaN

Gemma 4 12B LoRA
   conf  preds    share   acc   adj
0.9–1.0      0 0.000000   NaN   NaN
0.8–0.9      0 0.000000   NaN   NaN
0.7–0.8      0 0.000000   NaN   NaN
0

## Confusion matrices (test) — 4 panels

Same Greys, row-normalized style as every other confusion matrix in this
project; zero cells are left blank rather than showing "0.0% (0)"

In [50]:
PANEL_TITLES = ["ModernBERT", "T5Gemma 2", "Gemma 4 (fine-tuned)", "Gemma 4 (24-shot)"]

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=PANEL_TITLES,
                    horizontal_spacing=0.15, vertical_spacing=0.10)
fig.update_layout(coloraxis=dict(colorscale="Greys", cmin=0, cmax=1,
                                 colorbar=dict(title="Row-normalized")))

for k, c in enumerate(CONDITIONS):
    row, col = k // 2 + 1, k % 2 + 1
    d = load_preds(c["run_dir"], c["test_file"])
    counts = confusion_matrix(d["gold"], d["pred"], labels=list(range(NUM_CLASSES)))
    row_sums = counts.sum(axis=1, keepdims=True).flatten()
    with np.errstate(divide="ignore", invalid="ignore"):
        norm = np.nan_to_num(np.divide(counts, counts.sum(axis=1, keepdims=True),
                                       where=counts.sum(axis=1, keepdims=True) != 0))

    annot = np.empty_like(counts, dtype=object)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            annot[i, j] = f"{norm[i, j] * 100:.0f}%" if counts[i, j] else ""

    y_labels = [f"{lvl} (n={n})" for lvl, n in zip(LEVELS, row_sums)]

    fig.add_trace(
        go.Heatmap(
            z=norm, x=LEVELS, y=y_labels,
            text=annot, texttemplate="%{text}",
            textfont=dict(size=36),
            hoverongaps=False, coloraxis="coloraxis",
        ),
        row=row, col=col,
    )
    fig.update_xaxes(title_text="Predicted", tickfont=dict(size=36), row=row, col=col)
    fig.update_yaxes(title_text="True", tickfont=dict(size=36),
                     autorange="reversed", row=row, col=col)

for ann in fig.layout.annotations:
    ann.font.size = 36

fig.update_layout(
    font=dict(family="Arial", size=36, color="black"),
    margin=dict(l=150, r=110, t=100, b=90),
)

fig.show()

TARGET_WIDTH_IN = 6.9      # set to your ACTUAL insertion width in the manuscript
DPI = 300
W = int(TARGET_WIDTH_IN * DPI)
H = int(W * 1.05)

fig.write_image(os.path.join(OUT_DIR, "confusion_matrices_test.svg"), width=W, height=H, scale=1)
fig.write_image(os.path.join(OUT_DIR, "confusion_matrices_test.png"), width=W, height=H, scale=6)